# 13 Llm Investigative Brief Generator

**Project:** Insurance Fraud Detection Assistant

**Notebook:** `13-llm-investigative-brief-generator.ipynb`

In [1]:
# ==========================================
# Notebook 13
# LLM Investigative Brief Generator
# ==========================================

import pandas as pd
import numpy as np
import json

from transformers import pipeline

In [2]:
claims_df = pd.read_csv("../data/insurance_claims.csv")

cross_encoder_df = pd.read_csv("../data/cross_encoder_verification.csv")

template_fraud_df = pd.read_csv("../data/template_fraud_candidates.csv")

ring_candidates_df = pd.read_csv("../data/semantic_ring_candidates.csv")

fraud_risk_df = pd.read_csv("../data/fraud_risk_scores.csv")

In [3]:
claims_df.head()

,claim_id,claimant_name,vehicle_vin,mechanic_shop,clinic_name,lawyer,claimant_statement,police_report,adjuster_notes,medical_bill,fraud_label
0,CLM001,Wendy Holland,FqH15433919443,Rapid Auto Repair,Care First Clinic,Smith & Associates,A vehicle rear-ended me while I was waiting at...,Witnesses confirmed another driver caused the ...,Section international though many movement.,5072,0
1,CLM002,Douglas Lara,acF49501195178,Rapid Auto Repair,Wellness Recovery Center,Anderson Legal,The vehicle changed lanes unexpectedly and hit...,Police observed damage consistent with reporte...,Budget Mrs part spend middle threat smile incr...,1541,0
2,CLM003,Chloe Murphy,xeQ24677572737,Rapid Auto Repair,Care First Clinic,Justice Partners,I was stopped at a red light when another vehi...,Accident report indicates claimant followed tr...,Similar never box line.,20226,1
3,CLM004,Jodi Reynolds MD,sPL40843321198,Rapid Auto Repair,Wellness Recovery Center,Justice Partners,The vehicle changed lanes unexpectedly and hit...,Police observed damage consistent with reporte...,Section season nor political bank.,7723,0
4,CLM005,Elizabeth Patel,mmr35740163797,Prime Vehicle Repair,Wellness Recovery Center,Smith & Associates,I was driving through an intersection when ano...,Witnesses confirmed another driver caused the ...,Kind compare across audience society.,23376,0


In [4]:
llm = pipeline("text-generation", model="google/flan-t5-base")

Device set to use cpu
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausalLM', 'FlexOlmoF

In [5]:
def collect_claim_evidence(claim_id):

    claim = claims_df[claims_df["claim_id"] == claim_id].iloc[0]

    fraud_score = fraud_risk_df[fraud_risk_df["claim_id"] == claim_id].iloc[0]

    template_matches = template_fraud_df[
        (template_fraud_df["claim_1"] == claim_id)
        | (template_fraud_df["claim_2"] == claim_id)
    ]

    ring_matches = ring_candidates_df[
        (ring_candidates_df["claim_1"] == claim_id)
        | (ring_candidates_df["claim_2"] == claim_id)
    ]

    cross_match = cross_encoder_df[cross_encoder_df["claim_id"] == claim_id]

    verification_score = None

    if len(cross_match) > 0:

        verification_score = float(cross_match.iloc[0]["normalized_score"])

    evidence = {
        "claim_id": claim_id,
        "claimant_statement": claim["claimant_statement"],
        "police_report": claim["police_report"],
        "adjuster_notes": claim["adjuster_notes"],
        "fraud_risk_index": float(fraud_score["fraud_risk_index"]),
        "risk_category": fraud_score["risk_category"],
        "verification_score": verification_score,
        "template_matches": len(template_matches),
        "ring_matches": len(ring_matches),
    }

    return evidence

In [6]:
collect_claim_evidence("CLM001")

{'claim_id': 'CLM001',
 'claimant_statement': 'A vehicle rear-ended me while I was waiting at a traffic signal.',
 'police_report': 'Witnesses confirmed another driver caused the accident.',
 'adjuster_notes': 'Section international though many movement.',
 'fraud_risk_index': 75.89,
 'risk_category': 'High',
 'verification_score': 0.1821472668914981,
 'template_matches': 1,
 'ring_matches': 2}

In [7]:
def build_investigation_prompt(evidence):

    prompt = f"""

You are a Senior Insurance Fraud Investigator.

Analyze the following insurance claim.

Claim ID:
{evidence['claim_id']}

Fraud Risk Index:
{evidence['fraud_risk_index']}

Risk Category:
{evidence['risk_category']}

Cross Encoder Verification Score:
{evidence['verification_score']}

Template Fraud Matches:
{evidence['template_matches']}

Collusion Ring Matches:
{evidence['ring_matches']}

Claimant Statement:
{evidence['claimant_statement']}

Police Report:
{evidence['police_report']}

Adjuster Notes:
{evidence['adjuster_notes']}

Generate:

1. Executive Summary
2. Key Fraud Indicators
3. Contradictions Found
4. Investigation Recommendation

"""

    return prompt

In [8]:
def generate_investigation_brief(claim_id):

    evidence = collect_claim_evidence(claim_id)

    prompt = build_investigation_prompt(evidence)

    response = llm(prompt, max_new_tokens=250, do_sample=False)

    return {
        "claim_id": claim_id,
        "fraud_risk_index": evidence["fraud_risk_index"],
        "risk_category": evidence["risk_category"],
        "investigation_brief": response[0]["generated_text"],
    }

In [9]:
report = generate_investigation_brief("CLM014")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [10]:
print(report["investigation_brief"])



You are a Senior Insurance Fraud Investigator.

Analyze the following insurance claim.

Claim ID:
CLM014

Fraud Risk Index:
65.0

Risk Category:
High

Cross Encoder Verification Score:
0.0

Template Fraud Matches:
1

Collusion Ring Matches:
0

Claimant Statement:
I was traveling within the speed limit when the accident occurred.

Police Report:
Traffic camera footage supports claimant statement.

Adjuster Notes:
Treatment design first ok find audience.

Generate:

1. Executive Summary
2. Key Fraud Indicators
3. Contradictions Found
4. Investigation Recommendation




In [11]:
all_reports = []

for claim_id in claims_df["claim_id"]:

    report = generate_investigation_brief(claim_id)

    all_reports.append(report)

In [12]:
reports_df = pd.DataFrame(all_reports)

In [13]:
reports_df.head()

,claim_id,fraud_risk_index,risk_category,investigation_brief
0,CLM001,75.89,High,\n\nYou are a Senior Insurance Fraud Investiga...
1,CLM002,57.44,Medium,\n\nYou are a Senior Insurance Fraud Investiga...
2,CLM003,64.40,High,\n\nYou are a Senior Insurance Fraud Investiga...
3,CLM004,57.44,Medium,\n\nYou are a Senior Insurance Fraud Investiga...
4,CLM005,25.00,Low,\n\nYou are a Senior Insurance Fraud Investiga...


In [14]:
reports_df.sort_values(by="fraud_risk_index", ascending=False).head(10)

,claim_id,fraud_risk_index,risk_category,investigation_brief
6,CLM007,89.76,Critical,\n\nYou are a Senior Insurance Fraud Investiga...
8,CLM009,84.71,Critical,\n\nYou are a Senior Insurance Fraud Investiga...
0,CLM001,75.89,High,\n\nYou are a Senior Insurance Fraud Investiga...
7,CLM008,74.33,High,\n\nYou are a Senior Insurance Fraud Investiga...
13,CLM014,65.00,High,\n\nYou are a Senior Insurance Fraud Investiga...
2,CLM003,64.40,High,\n\nYou are a Senior Insurance Fraud Investiga...
12,CLM013,64.36,High,\n\nYou are a Senior Insurance Fraud Investiga...
1,CLM002,57.44,Medium,\n\nYou are a Senior Insurance Fraud Investiga...
3,CLM004,57.44,Medium,\n\nYou are a Senior Insurance Fraud Investiga...
9,CLM010,50.00,Medium,\n\nYou are a Senior Insurance Fraud Investiga...


In [15]:
def generate_json_brief(claim_id):

    evidence = collect_claim_evidence(claim_id)

    return {
        "claim_id": claim_id,
        "fraud_risk_index": evidence["fraud_risk_index"],
        "risk_category": evidence["risk_category"],
        "verification_score": evidence["verification_score"],
        "template_matches": evidence["template_matches"],
        "ring_matches": evidence["ring_matches"],
        "recommended_action": (
            "Escalate to SIU" if evidence["fraud_risk_index"] >= 80 else "Manual Review"
        ),
    }

In [16]:
generate_json_brief("CLM014")

{'claim_id': 'CLM014',
 'fraud_risk_index': 65.0,
 'risk_category': 'High',
 'verification_score': 0.0,
 'template_matches': 1,
 'ring_matches': 0,
 'recommended_action': 'Manual Review'}

In [17]:
json_reports = []

for claim_id in claims_df["claim_id"]:

    json_reports.append(generate_json_brief(claim_id))

In [18]:
json_reports_df = pd.DataFrame(json_reports)

In [19]:
json_reports_df.head()

,claim_id,fraud_risk_index,risk_category,verification_score,template_matches,ring_matches,recommended_action
0,CLM001,75.89,High,0.182147,1,2,Manual Review
1,CLM002,57.44,Medium,0.351212,1,1,Manual Review
2,CLM003,64.40,High,0.111990,0,2,Manual Review
3,CLM004,57.44,Medium,0.351212,1,1,Manual Review
4,CLM005,25.00,Low,1.000000,3,0,Manual Review


In [20]:
json_reports_df.sort_values(by="fraud_risk_index", ascending=False)

,claim_id,fraud_risk_index,risk_category,verification_score,template_matches,ring_matches,recommended_action
6,CLM007,89.76,Critical,0.204733,3,4,Escalate to SIU
8,CLM009,84.71,Critical,0.105872,1,3,Escalate to SIU
0,CLM001,75.89,High,0.182147,1,2,Manual Review
7,CLM008,74.33,High,0.213401,1,2,Manual Review
13,CLM014,65.00,High,0.000000,1,0,Manual Review
2,CLM003,64.40,High,0.111990,0,2,Manual Review
12,CLM013,64.36,High,0.012878,1,0,Manual Review
1,CLM002,57.44,Medium,0.351212,1,1,Manual Review
3,CLM004,57.44,Medium,0.351212,1,1,Manual Review
9,CLM010,50.00,Medium,1.000000,3,3,Manual Review


In [21]:
reports_df.to_csv("../data/llm_investigation_briefs.csv", index=False)

In [22]:
json_reports_df.to_csv("../data/structured_fraud_reports.csv", index=False)

In [23]:
investigation_queue = json_reports_df[json_reports_df["fraud_risk_index"] >= 60]

In [24]:
investigation_queue

,claim_id,fraud_risk_index,risk_category,verification_score,template_matches,ring_matches,recommended_action
0,CLM001,75.89,High,0.182147,1,2,Manual Review
2,CLM003,64.40,High,0.111990,0,2,Manual Review
6,CLM007,89.76,Critical,0.204733,3,4,Escalate to SIU
7,CLM008,74.33,High,0.213401,1,2,Manual Review
8,CLM009,84.71,Critical,0.105872,1,3,Escalate to SIU
12,CLM013,64.36,High,0.012878,1,0,Manual Review
13,CLM014,65.00,High,0.000000,1,0,Manual Review


In [25]:
investigation_queue.to_csv("../data/fraud_investigation_queue.csv", index=False)